# Combine model training summaries

Combines only the `*_architecture_summary.csv` files under each model's `Outputs` folder. The `train_num` column is the run number parsed from the output folder (for example, `Transformer_train_11` gives `11`); it is a run identifier, not a count of training examples. MLP runs also retain their architecture/configuration subfolder. Since architecture summaries have different columns, the combined CSV preserves the union of their columns and identifies each source with `source_summary`.

In [1]:
from pathlib import Path
import re
import pandas as pd


def locate_models_dir() -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name.lower() == "models":
        return cwd
    for parent in (cwd, *cwd.parents):
        candidate = parent / "Models"
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not find the repository's Models directory from {cwd}. "
        "Run this notebook from the repository or its Models folder."
    )


models_dir = locate_models_dir()
train_pattern = re.compile(r"train[_ ]*(\d+)", flags=re.IGNORECASE)
summary_files = sorted(
    path
    for path in models_dir.glob("*/Outputs/**/*.csv")
    if path.name.lower().endswith("_architecture_summary.csv")
)

if not summary_files:
    raise FileNotFoundError(
        f"No *_architecture_summary.csv files found under {models_dir}."
    )

frames = []
for path in summary_files:
    relative_to_model = path.relative_to(models_dir)
    model_family = relative_to_model.parts[0]
    output_index = relative_to_model.parts.index("Outputs")
    run_parts = relative_to_model.parts[output_index + 1 : -1]
    train_match = next(
        ((index, train_pattern.search(part)) for index, part in enumerate(run_parts)
         if train_pattern.search(part)),
        None,
    )
    if train_match is None:
        raise ValueError(f"Could not parse a train number from output path: {path}")

    train_folder_index, match = train_match
    run_folder = run_parts[train_folder_index]
    config_parts = run_parts[train_folder_index + 1 :]
    run_config = Path(*config_parts).as_posix() if config_parts else "(default)"

    frame = pd.read_csv(path)
    frame.insert(0, "model_family", model_family)
    frame.insert(1, "train_num", int(match.group(1)))
    frame.insert(2, "train_folder", run_folder)
    frame.insert(3, "run_config", run_config)
    frame.insert(4, "source_summary", relative_to_model.as_posix())
    frames.append(frame)

combined = pd.concat(frames, ignore_index=True, sort=False)
output_csv = models_dir / "all_model_architecture_summaries.csv"
combined.to_csv(output_csv, index=False)

print(f"Combined {len(summary_files)} summaries into {output_csv}")
print(f"Combined rows: {len(combined)} | Columns: {len(combined.columns)}")
print("Summaries by model family and train number:")
display(
    combined.groupby(["model_family", "train_num"], dropna=False)
    .size()
    .rename("summary_rows")
    .reset_index()
)


Combined 41 summaries into E:\NMR_project\NMRPrediction\Models\all_model_architecture_summaries.csv
Combined rows: 41 | Columns: 132
Summaries by model family and train number:


,model_family,train_num,summary_rows
0,CNN,1,1
1,CNN,2,1
2,CNN,3,1
3,CNN,4,1
4,CNN,5,1
5,CNN,6,1
6,CNN,7,1
7,CNN,8,1
8,CNN,9,1
9,CNN,10,1
